# Task 03 — Δz distribution of the hard set

## 🌌 Macrocosm photo-z — outlier study (tasks 2026-06-16)

**✅ GIVEN (do NOT re-derive this):** we already ran a **400k 5-fold cross-validation** with three
models (HGB, RF, MLP) and collected the galaxies that *all three* predict catastrophically out-of-fold
(|Δz/(1+z)| > 0.05). There are **6,974** such **hard** galaxies. Their objids are in
**`../hard_objids.csv`**. Treat this hard set as a **fixed input** — your job is to characterize / act
on it, not to re-find it from 400k.

**Catalog:** `gs://macrocosm-lewagon/data/sample_v1/catalog_v1.parquet` (600k rows, 55 columns).
The SDSS **`-9999` sentinel** means "not measured" — always clean it to NaN first.

**Metric:** `σ_MAD = 1.4826 · median(|Δz − median(Δz)|)`, with `Δz = (z_pred − z_true)/(1+z)`;
an **outlier** is `|Δz| > 0.05`. σ_MAD is the headline metric, report **outlier rate** alongside it.

---

Given the 6,974 hard galaxies, look at *how* a model fails on them. Is it a one-sided bias (correctable)
or two-sided scatter? Compare the hard subset against the full population.

In [ ]:
# === shared setup: load catalog, clean -9999, build the 16 features ===
import pandas as pd, numpy as np

CATALOG = "gs://macrocosm-lewagon/data/sample_v1/catalog_v1.parquet"
# Colab: from google.colab import auth; auth.authenticate_user()
# or download once: !gcloud storage cp $CATALOG . , then CATALOG = "catalog_v1.parquet"

FEATS = ["dered_u","dered_g","dered_r","dered_i","dered_z","g-r","u-g","r-i","i-z",
         "log_expRad_r","log_deVRad_r","log_petroRad_r","log_petroR50_r","log_petroR90_r",
         "fracDeV_r","conc_r"]

def load_features(path=CATALOG, n=None, seed=0):
    """Load catalog, clean the -9999 sentinel, build colors / log-sizes / conc.
    Returns (D, cat): D = features+redshift+objid (optionally subsampled), cat = full cleaned catalog."""
    cat = pd.read_parquet(path)
    num = cat.select_dtypes("number").columns
    cat[num] = cat[num].mask(cat[num] <= -100)                       # clean SDSS -9999
    for a, b in [("u","g"),("g","r"),("r","i"),("i","z")]:
        cat[f"{a}-{b}"] = (cat[f"dered_{a}"] - cat[f"dered_{b}"]).clip(-1, 4)
    for s in ["expRad_r","deVRad_r","petroRad_r","petroR50_r","petroR90_r"]:
        cat["log_"+s] = np.log1p(cat[s].clip(lower=0))
    cat["conc_r"] = cat["petroR90_r"] / cat["petroR50_r"].replace(0, np.nan)
    D = cat[FEATS + ["redshift","objid"]].replace([np.inf,-np.inf], np.nan).dropna()
    if n:
        D = D.sample(n, random_state=seed).reset_index(drop=True)
    return D, cat

def smad(dz): return 1.4826 * np.median(np.abs(dz - np.median(dz)))

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    dz = (y_pred - y_true) / (1 + y_true)
    return {"MAE": round(float(np.mean(np.abs(y_pred-y_true))), 5),
            "sigma_MAD": round(float(smad(dz)), 5),
            "outlier_rate": round(float(np.mean(np.abs(dz) > 0.05)), 5)}

# the GIVEN hard set (6,974 objids from the 400k 5-fold CV)
HARD = set(pd.read_csv("../hard_objids.csv")["objid"])
print(f"hard set: {len(HARD)} galaxies")

❓ **Question (OOF predictions)** ❓

👇 Train one strong model (RF or MLP) out-of-fold on 100k (`seed=1`, `KFold(3, ...,42)`), so every galaxy has an OOF prediction. Compute Δz for all galaxies.

In [ ]:
# YOUR CODE HERE
D, cat = load_features(n=100000, seed=1)


❓ **Question (hard vs full)** ❓

👇 Split by `D['objid'].isin(HARD)`. Plot the Δz histogram of the hard subset overlaid on the full population (mark ±0.05). Report the hard-set Δz median and the fraction with Δz<-0.05 vs >0.05.

In [ ]:
# YOUR CODE HERE
import matplotlib.pyplot as plt
is_hard = D['objid'].isin(HARD)


❓ **Question (verdict)** ❓

👇 Is the hard Δz one-sided (a bias) or two-sided (scatter)? What does that imply for bias-correction?

In [ ]:
# YOUR CODE HERE


## 📝 Write your report

In [ ]:
# === write_report: run this after you've filled in your results, it generates report.md ===
def write_report(title, results: dict, conclusion: str, path="report.md"):
    lines = [f"# {title}", "", "_Auto-generated by task.ipynb — fill `results` and `conclusion` above._", "",
             "## Results", ""]
    for k, v in results.items():
        lines.append(f"- **{k}**: {v}")
    lines += ["", "## Conclusion", "", conclusion, ""]
    with open(path, "w") as f:
        f.write("\n".join(lines))
    print("wrote", path)

In [ ]:
write_report("Task 03 — Δz of the hard set",
             {"hard_dz_median": "FILL IN", "frac_under_-0.05": "FILL IN", "frac_over_0.05": "FILL IN"},
             "FILL IN: one-sided or two-sided? can a bias-correction fix it?")

In [ ]:
# === Commit & push your results (run last; make sure you are on branch 2026.6.16) ===
# First time: git pull origin 2026.6.16   to get the latest.
!git add task.ipynb report.md && git commit -m "03-dz-distribution task" && git push origin 2026.6.16